In [1]:
import requests
import datetime
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor
from collections import deque

In [2]:
# API Key (Replace with your actual API key)
API_KEY = 'ksnxCfuMd8YcYtNVDrqZKBY0aZeDMLX8'

# API Rate Limit Constants
API_LIMIT = 300  # Max API calls per minute
WINDOW_SIZE = 60  # Seconds (1 minute)

# Track request timestamps
request_times = deque()

def rate_limit():
    """Enforces the 300 API calls per minute limit."""
    global request_times

    # Remove timestamps older than 60 seconds
    now = time.time()
    while request_times and now - request_times[0] > WINDOW_SIZE:
        request_times.popleft()

    # If we're at the limit, wait until a slot opens
    if len(request_times) >= API_LIMIT:
        wait_time = WINDOW_SIZE - (now - request_times[0])
        print(f"Rate limit reached. Sleeping for {wait_time:.2f} seconds...")
        time.sleep(wait_time)
    
    # Register new API request
    request_times.append(time.time())


def get_all_tickers():
    """Fetches a list of all investable stocks and indexes from FMP API."""
    stock_url = f"https://financialmodelingprep.com/api/v3/stock/list?apikey={API_KEY}"
    index_url = f"https://financialmodelingprep.com/api/v3/symbol/available-indexes?apikey={API_KEY}"

    try:
        rate_limit()
        stock_response = requests.get(stock_url).json()
        rate_limit()
        index_response = requests.get(index_url).json()

        stock_tickers = [item['symbol'] for item in stock_response]
        index_tickers = [item['symbol'] for item in index_response]

        return stock_tickers + index_tickers
    except Exception as e:
        print("Error fetching tickers:", e)
        return []


def get_sd(ticker):
    """Fetches standard deviation for a stock over 1Y, 5Y, and 10Y."""
    base_url = 'https://financialmodelingprep.com/api/v3/technical_indicator/1day/'
    periods = {"1Y": 252, "5Y": 1260, "10Y": 2520}
    sd_values = {}

    for key, period in periods.items():
        url = f"{base_url}{ticker}?type=standardDeviation&period={period}&apikey={API_KEY}"
        try:
            rate_limit()
            response = requests.get(url)
            data = response.json()

            if isinstance(data, list) and len(data) > 0:
                sd_values[key] = round(data[0].get('standardDeviation', None), 4)
            else:
                sd_values[key] = None
        except Exception as e:
            print(f"Error fetching SD for {ticker}: {e}")
            sd_values[key] = None

    return sd_values


def get_cagr(ticker):
    """Fetches CAGR for a stock over 1Y, 5Y, and 10Y."""
    base_url = 'https://financialmodelingprep.com/api/v3/historical-price-full/'
    years_list = [1, 5, 10]
    cagr_values = {}

    for years in years_list:
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=years * 365)

        url = f"{base_url}{ticker}?from={start_date}&to={end_date}&apikey={API_KEY}"
        try:
            rate_limit()
            response = requests.get(url)
            data = response.json()

            if "historical" in data and len(data["historical"]) > 0:
                historical_data = sorted(data["historical"], key=lambda x: x["date"])
                P_start = historical_data[0]["close"]
                P_end = historical_data[-1]["close"]

                cagr = ((P_end / P_start) ** (1 / years)) - 1
                cagr_values[f"{years}Y"] = round(cagr * 100, 2)
            else:
                cagr_values[f"{years}Y"] = None
        except Exception as e:
            print(f"Error fetching CAGR for {ticker}: {e}")
            cagr_values[f"{years}Y"] = None

    return cagr_values


def get_stock_data(ticker):
    """Combines Standard Deviation and CAGR data into a single dictionary per stock."""
    try:
        sd_data = get_sd(ticker)
        cagr_data = get_cagr(ticker)

        return {
            "Ticker": ticker,
            "1Y SD": sd_data["1Y"],
            "5Y SD": sd_data["5Y"],
            "10Y SD": sd_data["10Y"],
            "1Y CAGR": cagr_data["1Y"],
            "5Y CAGR": cagr_data["5Y"],
            "10Y CAGR": cagr_data["10Y"]
        }
    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        return None


def get_multiple_stocks_data(tickers, workers=50):
    """Fetches SD and CAGR for multiple stocks using multithreading while respecting rate limits."""
    total_tickers = len(tickers)
    all_data = []

    print(f"Processing {total_tickers} tickers using {workers} threads...")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = list(executor.map(get_stock_data, tickers))

    # Filter out None values (failed requests)
    all_data = [result for result in results if result]

    # Convert to DataFrame
    df = pd.DataFrame(all_data)

    # Save interim results
    df.to_csv("all_stocks_sd_cagr.csv", index=False)

    return df

In [3]:
# **Step 1: Get All Investable Stocks & Indexes**
all_tickers = get_all_tickers()

# **Step 2: Process API Calls in Parallel with Rate Limiting**
df = get_multiple_stocks_data(all_tickers, workers=50)

# **Step 3: Save Final Data**
df.to_csv("all_stocks_sd_cagr.csv", index=False)

# **Step 4: Display Sample Output**
print(df.head())

Processing 85099 tickers using 50 threads...
Rate limit reached. Sleeping for 32.04 seconds...Rate limit reached. Sleeping for 32.06 seconds...

Rate limit reached. Sleeping for 32.01 seconds...
Rate limit reached. Sleeping for 31.96 seconds...
Rate limit reached. Sleeping for 31.94 seconds...
Rate limit reached. Sleeping for 31.92 seconds...
Rate limit reached. Sleeping for 31.86 seconds...
Rate limit reached. Sleeping for 31.71 seconds...
Rate limit reached. Sleeping for 31.51 seconds...
Rate limit reached. Sleeping for 31.31 seconds...
Rate limit reached. Sleeping for 31.21 seconds...
Rate limit reached. Sleeping for 31.19 seconds...
Rate limit reached. Sleeping for 31.17 seconds...
Rate limit reached. Sleeping for 31.05 seconds...
Rate limit reached. Sleeping for 30.87 seconds...
Rate limit reached. Sleeping for 30.82 seconds...
Rate limit reached. Sleeping for 30.64 seconds...
Rate limit reached. Sleeping for 30.39 seconds...
Rate limit reached. Sleeping for 30.28 seconds...
Rate 

Rate limit reached. Sleeping for 0.17 seconds...
Rate limit reached. Sleeping for 0.16 seconds...
Rate limit reached. Sleeping for 0.15 seconds...
Rate limit reached. Sleeping for 0.14 seconds...
Rate limit reached. Sleeping for 0.11 seconds...
Rate limit reached. Sleeping for 0.03 seconds...
Rate limit reached. Sleeping for 0.02 seconds...
Rate limit reached. Sleeping for 0.02 seconds...
Rate limit reached. Sleeping for 0.01 seconds...
Rate limit reached. Sleeping for 0.06 seconds...
Rate limit reached. Sleeping for 0.03 seconds...
Rate limit reached. Sleeping for 0.02 seconds...
Rate limit reached. Sleeping for 0.01 seconds...
Rate limit reached. Sleeping for 0.12 seconds...
Rate limit reached. Sleeping for 0.11 seconds...
Rate limit reached. Sleeping for 0.09 seconds...
Rate limit reached. Sleeping for 0.08 seconds...
Rate limit reached. Sleeping for 0.05 seconds...
Rate limit reached. Sleeping for 0.05 seconds...
Rate limit reached. Sleeping for 0.02 seconds...
Rate limit reached. 

Rate limit reached. Sleeping for 28.33 seconds...
Rate limit reached. Sleeping for 28.30 seconds...
Rate limit reached. Sleeping for 0.19 seconds...
Rate limit reached. Sleeping for 0.17 seconds...
Rate limit reached. Sleeping for 0.16 seconds...
Rate limit reached. Sleeping for 0.12 seconds...
Rate limit reached. Sleeping for 0.12 seconds...
Rate limit reached. Sleeping for 0.10 seconds...
Rate limit reached. Sleeping for 0.09 seconds...
Rate limit reached. Sleeping for 0.04 seconds...
Rate limit reached. Sleeping for 0.04 seconds...
Rate limit reached. Sleeping for 0.03 seconds...
Rate limit reached. Sleeping for 0.02 seconds...
Rate limit reached. Sleeping for 0.02 seconds...
Rate limit reached. Sleeping for 0.01 seconds...
Rate limit reached. Sleeping for 0.00 seconds...
Rate limit reached. Sleeping for 0.00 seconds...
Rate limit reached. Sleeping for 0.32 seconds...
Rate limit reached. Sleeping for 0.30 seconds...
Rate limit reached. Sleeping for 0.29 seconds...
Rate limit reached

Rate limit reached. Sleeping for 0.23 seconds...
Rate limit reached. Sleeping for 0.09 seconds...
Rate limit reached. Sleeping for 0.06 seconds...
Rate limit reached. Sleeping for 0.06 seconds...
Rate limit reached. Sleeping for 0.08 seconds...
Rate limit reached. Sleeping for 0.02 seconds...
Rate limit reached. Sleeping for 31.01 seconds...
Rate limit reached. Sleeping for 30.54 seconds...
Rate limit reached. Sleeping for 29.57 seconds...
Rate limit reached. Sleeping for 29.34 seconds...
Rate limit reached. Sleeping for 28.79 seconds...
Rate limit reached. Sleeping for 28.66 seconds...
Rate limit reached. Sleeping for 28.62 seconds...
Rate limit reached. Sleeping for 28.59 seconds...
Rate limit reached. Sleeping for 28.43 seconds...
Rate limit reached. Sleeping for 28.36 seconds...
Rate limit reached. Sleeping for 28.33 seconds...
Rate limit reached. Sleeping for 28.30 seconds...
Rate limit reached. Sleeping for 28.25 seconds...
Rate limit reached. Sleeping for 28.22 seconds...
Rate l

Rate limit reached. Sleeping for 0.91 seconds...
Rate limit reached. Sleeping for 0.90 seconds...
Rate limit reached. Sleeping for 0.90 seconds...
Rate limit reached. Sleeping for 0.89 seconds...
Rate limit reached. Sleeping for 0.87 seconds...
Rate limit reached. Sleeping for 0.85 seconds...
Rate limit reached. Sleeping for 0.84 seconds...
Rate limit reached. Sleeping for 0.83 seconds...
Rate limit reached. Sleeping for 0.82 seconds...
Rate limit reached. Sleeping for 0.80 seconds...
Rate limit reached. Sleeping for 0.78 seconds...
Rate limit reached. Sleeping for 0.77 seconds...
Rate limit reached. Sleeping for 0.75 seconds...
Rate limit reached. Sleeping for 0.71 seconds...
Rate limit reached. Sleeping for 0.70 seconds...
Rate limit reached. Sleeping for 0.69 seconds...
Rate limit reached. Sleeping for 0.68 seconds...
Rate limit reached. Sleeping for 0.67 seconds...
Rate limit reached. Sleeping for 0.64 seconds...
Rate limit reached. Sleeping for 0.63 seconds...
Rate limit reached. 

...

Rate limit reached. Sleeping for 0.21 seconds...
Rate limit reached. Sleeping for 6.31 seconds...
Rate limit reached. Sleeping for 5.66 seconds...
Rate limit reached. Sleeping for 2.92 seconds...
Rate limit reached. Sleeping for 2.70 seconds...
Rate limit reached. Sleeping for 2.65 seconds...
Rate limit reached. Sleeping for 2.65 seconds...
Rate limit reached. Sleeping for 2.64 seconds...
Rate limit reached. Sleeping for 2.63 seconds...
Rate limit reached. Sleeping for 2.59 seconds...
Rate limit reached. Sleeping for 2.59 seconds...
Rate limit reached. Sleeping for 2.52 seconds...
Rate limit reached. Sleeping for 2.46 seconds...
Rate limit reached. Sleeping for 2.46 seconds...
Rate limit reached. Sleeping for 2.45 seconds...
Rate limit reached. Sleeping for 2.44 seconds...
Rate limit reached. Sleeping for 2.43 seconds...
Rate limit reached. Sleeping for 2.42 seconds...
Rate limit reached. Sleeping for 2.41 seconds...
Rate limit reached. Sleeping for 2.41 seconds...
Rate limit reached. 

Rate limit reached. Sleeping for 0.83 seconds...
Rate limit reached. Sleeping for 0.82 seconds...
Rate limit reached. Sleeping for 0.81 seconds...
Rate limit reached. Sleeping for 0.77 seconds...
Rate limit reached. Sleeping for 0.76 seconds...
Rate limit reached. Sleeping for 0.76 seconds...
Rate limit reached. Sleeping for 0.75 seconds...
Rate limit reached. Sleeping for 0.75 seconds...
Rate limit reached. Sleeping for 0.74 seconds...
Rate limit reached. Sleeping for 0.73 seconds...
Rate limit reached. Sleeping for 0.72 seconds...
Rate limit reached. Sleeping for 0.71 seconds...
Rate limit reached. Sleeping for 0.71 seconds...
Rate limit reached. Sleeping for 0.67 seconds...
Rate limit reached. Sleeping for 0.62 seconds...
Rate limit reached. Sleeping for 0.62 seconds...
Rate limit reached. Sleeping for 0.58 seconds...
Rate limit reached. Sleeping for 0.56 seconds...
Rate limit reached. Sleeping for 0.50 seconds...
Rate limit reached. Sleeping for 0.47 seconds...
Rate limit reached. 

Rate limit reached. Sleeping for 1.44 seconds...
Rate limit reached. Sleeping for 1.42 seconds...
Rate limit reached. Sleeping for 1.40 seconds...
Rate limit reached. Sleeping for 1.37 seconds...
Rate limit reached. Sleeping for 1.37 seconds...
Rate limit reached. Sleeping for 1.36 seconds...
Rate limit reached. Sleeping for 1.30 seconds...
Rate limit reached. Sleeping for 1.27 seconds...
Rate limit reached. Sleeping for 1.25 seconds...
Rate limit reached. Sleeping for 1.24 seconds...
Rate limit reached. Sleeping for 1.22 seconds...
Rate limit reached. Sleeping for 1.18 seconds...
Rate limit reached. Sleeping for 1.18 seconds...
Rate limit reached. Sleeping for 1.16 seconds...
Rate limit reached. Sleeping for 1.16 seconds...
Rate limit reached. Sleeping for 1.15 seconds...
Rate limit reached. Sleeping for 1.13 seconds...
Rate limit reached. Sleeping for 1.13 seconds...
Rate limit reached. Sleeping for 1.12 seconds...
Rate limit reached. Sleeping for 1.11 seconds...
Rate limit reached. 

Rate limit reached. Sleeping for 1.75 seconds...
Rate limit reached. Sleeping for 1.74 seconds...
Rate limit reached. Sleeping for 1.70 seconds...
Rate limit reached. Sleeping for 1.69 seconds...
Rate limit reached. Sleeping for 1.44 seconds...
Rate limit reached. Sleeping for 1.32 seconds...
Rate limit reached. Sleeping for 2.77 seconds...
Rate limit reached. Sleeping for 2.73 seconds...
Rate limit reached. Sleeping for 2.67 seconds...
Rate limit reached. Sleeping for 2.66 seconds...
Rate limit reached. Sleeping for 2.26 seconds...
Rate limit reached. Sleeping for 2.22 seconds...
Rate limit reached. Sleeping for 1.86 seconds...
Rate limit reached. Sleeping for 1.83 seconds...
Rate limit reached. Sleeping for 1.74 seconds...
Rate limit reached. Sleeping for 1.70 seconds...
Rate limit reached. Sleeping for 1.69 seconds...
Rate limit reached. Sleeping for 1.55 seconds...
Rate limit reached. Sleeping for 1.55 seconds...
Rate limit reached. Sleeping for 1.54 seconds...
Rate limit reached. 

KeyboardInterrupt: 